# 08 · Analisis Profundo de Comparacion Triple

**Objetivo:** Analizar sistematicamente DONDE y POR QUE falla el modelo en datos OOT,
comparando: **Resultado real** vs **Modelo XGBoost** vs **IA Previa Savia**.

## Analisis realizados
1. Exploracion de valores crudos de `ejecucion_glosa_aut_ia` y `estado_glosa_aut`
2. Matriz de acuerdo triple (3x3x3 = 27 celdas)
3. Analisis de errores por segmento (contrato, regimen, mes, NIT, valor)
4. Evolucion temporal dentro del periodo OOT
5. Comparacion de distribuciones de features: train vs OOT (PSI)
6. Deep-dive IA Previa: por que clasifica todo como "Auditada"?
7. Analisis de calibracion de probabilidades

**Entrada:** Test split + BD `system_savia` + OOT (ago-dic 2025)
**Salida:** `reports/evaluations/20260311_analisis_profundo.json` + figuras `ap_01..05`

In [1]:
# Celda 1 — Configuración, imports, conexión BD
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from pathlib import Path
import time
import json
import warnings
warnings.filterwarnings('ignore')

# ── Rutas ──────────────────────────────────────────────────────
BASE_DIR      = Path(r'D:\Users\jcardonr\Documents\eafit\Codigo')
DATA_DIR      = BASE_DIR / 'data' / 'processed'
EVAL_DIR      = BASE_DIR / 'reports' / 'evaluations'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

LABELS  = {0: 'Auditada', 1: 'Glosada', 2: 'Devuelta'}

# ── Conexión BD ────────────────────────────────────────────────
user, password, host, db_name, port = (
    "jcardonr", "6nxx2yVWaKXyiF1cZbfS*", "10.250.5.36", "system_savia", "3306"
)
engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_name}",
    connect_args={"connect_timeout": 60},
    pool_recycle=3600,
)

def query_to_df(sql_str: str) -> pd.DataFrame:
    """Ejecuta query SELECT y retorna DataFrame."""
    with engine.connect() as conn:
        result = conn.execute(text(sql_str))
        cols = list(result.keys())
        rows = result.fetchall()
        return pd.DataFrame(rows, columns=cols)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("✅ Conexión BD exitosa")

✅ Conexión BD exitosa


## 1. Pregunta clave: ¿Cómo validamos nuestro modelo?

**La validación ya se hizo en notebook 07** usando el target construido desde JOINs:
- `cm_auditoria_devoluciones` → Devuelta (clase 2)
- `cm_auditoria_motivos_glosas` → Glosada (clase 1)
- Ninguno de los dos → Auditada (clase 0)

**Pero surge la duda:** ¿`estado_factura` podría servir como ground truth alternativo?

- `estado_factura` tiene valores como `2=Auditada-Glosas`, `5=Devuelta`, etc.
- Es un **estado administrativo del flujo**, no necesariamente el resultado de auditoría
- Necesitamos verificar: ¿`estado_factura` coincide con nuestro target o no?

### Análisis en esta sección:
1. Distribución de `estado_factura` y su significado
2. Cruce `estado_factura` vs target real (JOINs) → ¿coinciden?
3. Conclusión: ¿usar `estado_factura` o no para validar?

In [ ]:
# Celda 2 — Distribución de estado_factura y cruce con target real
# ¿Qué valores tiene estado_factura y cómo se relacionan con nuestro target?

print("=" * 70)
print("DISTRIBUCIÓN de estado_factura (últimos 24 meses)")
print("=" * 70)

t0 = time.time()
df_ef = query_to_df("""
SELECT 
    f.estado_factura AS valor,
    COUNT(*) AS n_facturas,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM cm_facturas 
                               WHERE fecha_radicacion >= DATE_SUB(NOW(), INTERVAL 24 MONTH)), 2) AS pct
FROM cm_facturas f
WHERE f.fecha_radicacion >= DATE_SUB(NOW(), INTERVAL 24 MONTH)
GROUP BY f.estado_factura
ORDER BY n_facturas DESC
""")
print(f"  ({time.time()-t0:.1f}s)")

ESTADO_FACTURA_LABELS = {
    0: 'Radicado', 1: 'En Auditoría', 2: 'Auditada-Glosas', 
    3: 'Cierre Auditoría', 4: 'Conciliada', 5: 'Devuelta',
    6: 'Estado 6', 7: 'Estado 7', 8: 'Estado 8', 9: 'Estado 9',
    10: 'Estado 10', 11: 'Estado 11', 12: 'Estado 12', 13: 'Estado 13'
}
for _, row in df_ef.iterrows():
    label = ESTADO_FACTURA_LABELS.get(int(row['valor']), f"Estado {int(row['valor'])}")
    print(f"  {int(row['valor']):>3} ({label:>20}) → {int(row['n_facturas']):>10,} facturas ({row['pct']:>6.2f}%)")

print(f"\n{'='*70}")
print("CRUCE: estado_factura vs TARGET REAL (construido desde JOINs)")
print("=" * 70)
print("¿Coinciden? Si sí → estado_factura podría ser ground truth alternativo")
print("Si no → estado_factura es solo flujo administrativo, NO resultado de auditoría\n")

t0 = time.time()
df_cruce_ef = query_to_df("""
SELECT 
    f.estado_factura,
    CASE 
        WHEN d.id IS NOT NULL THEN 2
        WHEN EXISTS (SELECT 1 FROM cm_detalles cd 
                     INNER JOIN cm_auditoria_motivos_glosas g ON g.cm_detalles_id = cd.id 
                     WHERE cd.cm_facturas_id = f.id) THEN 1
        ELSE 0 
    END AS target_real,
    COUNT(*) AS n
FROM cm_facturas f
LEFT JOIN cm_auditoria_devoluciones d ON d.cm_facturas_id = f.id
WHERE f.fecha_radicacion >= '2024-03-01' AND f.fecha_radicacion < '2026-01-01'
GROUP BY f.estado_factura, target_real
ORDER BY f.estado_factura, target_real
""")
print(f"  Query ejecutada en {time.time()-t0:.1f}s\n")

# Pivot: filas = estado_factura, columnas = target_real
df_pivot_ef = df_cruce_ef.pivot_table(
    index='estado_factura', columns='target_real', values='n',
    fill_value=0, aggfunc='sum'
)
df_pivot_ef.columns = [LABELS.get(c, c) for c in df_pivot_ef.columns]
df_pivot_ef['Total'] = df_pivot_ef.sum(axis=1)

for col in ['Auditada', 'Glosada', 'Devuelta']:
    if col in df_pivot_ef.columns:
        df_pivot_ef[f'%_{col}'] = (df_pivot_ef[col] / df_pivot_ef['Total'] * 100).round(1)

df_pivot_ef.index = [f"{ESTADO_FACTURA_LABELS.get(idx, f'Estado {idx}')} (={idx})" for idx in df_pivot_ef.index]

print("── Tabla de contingencia: estado_factura vs target real ──\n")
print(df_pivot_ef.to_string())

print(f"\n{'='*70}")
print("INTERPRETACIÓN")
print("=" * 70)
print("""
Si estado_factura=2 ('Auditada-Glosas') correlaciona con target=1 (Glosada),
y estado_factura=5 ('Devuelta') correlaciona con target=2 (Devuelta),
entonces estado_factura PODRÍA ser un proxy del target.

Pero si estado_factura mezcla estados (ej: una factura 'Conciliada' puede
haber sido Glosada o Auditada), entonces NO es ground truth confiable.
""")

In [ ]:
# Celda 3 — Validación directa: cargar modelo y predecir sobre facturas con resultado conocido
# ESTA ES LA VALIDACIÓN REAL: tomo facturas cuyo resultado ya conozco,
# las paso por el modelo, y verifico que predice lo correcto.

import xgboost as xgb
import joblib

# ── Cargar modelo y features ──────────────────────────────────
MODEL_PATH = BASE_DIR / 'models' / 'artifacts' / '20260309_xgboost_v1.json'
FEATURE_LIST_PATH = DATA_DIR / '20260309_feature_list.txt'

model = xgb.XGBClassifier()
model.load_model(str(MODEL_PATH))
print(f"✅ Modelo cargado: {MODEL_PATH.name}")

with open(FEATURE_LIST_PATH) as f:
    feature_cols = [line.strip() for line in f if line.strip()]
print(f"✅ Features: {len(feature_cols)} columnas")

# ── Cargar dataset de test (resultado conocido) ───────────────
test_path = DATA_DIR / '20260309_test.parquet'
df_test = pd.read_parquet(test_path)
print(f"✅ Test set cargado: {len(df_test):,} facturas")
print(f"   Distribución real:")
for cls, label in LABELS.items():
    n = (df_test['target'] == cls).sum()
    print(f"   Clase {cls} ({label}): {n:,} ({n/len(df_test)*100:.1f}%)")

# ── Predecir ──────────────────────────────────────────────────
X_test = df_test[feature_cols]
y_true = df_test['target'].values

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

print(f"\n✅ Predicción completada sobre {len(y_pred):,} facturas")

In [ ]:
# Celda 4 — Métricas de validación y ejemplos concretos
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("=" * 70)
print("VALIDACIÓN: Predicción del modelo vs resultado REAL conocido")
print("=" * 70)

# Classification report completo
print("\n── Classification Report ──\n")
print(classification_report(y_true, y_pred, target_names=['Auditada', 'Glosada', 'Devuelta'], digits=4))

# F1-macro
f1_macro = f1_score(y_true, y_pred, average='macro')
print(f"F1-macro: {f1_macro:.4f}")

# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)
print("\n── Matriz de Confusión (filas=real, cols=predicción) ──\n")
print(f"{'':>15} {'Pred Audit':>12} {'Pred Glos':>12} {'Pred Dev':>12}")
for i, label in LABELS.items():
    print(f"  Real {label:>8} {cm[i,0]:>12,} {cm[i,1]:>12,} {cm[i,2]:>12,}")

# Matriz normalizada (porcentajes)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
print("\n── Matriz de Confusión NORMALIZADA (%) ──\n")
print(f"{'':>15} {'Pred Audit':>12} {'Pred Glos':>12} {'Pred Dev':>12}")
for i, label in LABELS.items():
    print(f"  Real {label:>8} {cm_norm[i,0]:>11.1f}% {cm_norm[i,1]:>11.1f}% {cm_norm[i,2]:>11.1f}%")

print(f"\n{'='*70}")
print("EJEMPLOS CONCRETOS de predicción correcta e incorrecta")
print("=" * 70)

df_test_pred = df_test.copy()
df_test_pred['pred'] = y_pred
df_test_pred['prob_auditada'] = y_proba[:, 0]
df_test_pred['prob_glosada'] = y_proba[:, 1]
df_test_pred['prob_devuelta'] = y_proba[:, 2]
df_test_pred['acierto'] = (df_test_pred['target'] == df_test_pred['pred'])

# Resumen por clase
for cls, label in LABELS.items():
    sub = df_test_pred[df_test_pred['target'] == cls]
    n_correct = sub['acierto'].sum()
    n_total = len(sub)
    print(f"\n  Clase {cls} ({label}): {n_correct:,}/{n_total:,} correctas ({n_correct/n_total*100:.1f}%)")
    
    # 3 ejemplos correctos
    correctos = sub[sub['acierto']].head(3)
    if len(correctos) > 0:
        print(f"    ✅ Ejemplos correctos:")
        for _, row in correctos.iterrows():
            print(f"       factura_id={int(row['factura_id'])}  "
                  f"prob=[A:{row['prob_auditada']:.2f} G:{row['prob_glosada']:.2f} D:{row['prob_devuelta']:.2f}]")
    
    # 3 ejemplos incorrectos
    incorrectos = sub[~sub['acierto']].head(3)
    if len(incorrectos) > 0:
        print(f"    ❌ Ejemplos incorrectos (predijo {LABELS.get(int(incorrectos.iloc[0]['pred']), '?')}):")
        for _, row in incorrectos.iterrows():
            print(f"       factura_id={int(row['factura_id'])}  pred={LABELS[int(row['pred'])]}  "
                  f"prob=[A:{row['prob_auditada']:.2f} G:{row['prob_glosada']:.2f} D:{row['prob_devuelta']:.2f}]")

In [ ]:
# Celda 5 — Cruce por estado_glosa_aut: ¿Qué pasa con las facturas procesadas por el bot?

print("=" * 70)
print("CRUCE: estado_glosa_aut vs TARGET REAL")
print("=" * 70)
print("Estados: 0=Pendiente, 1=bot en proceso, 2=bot ejecutado,")
print("         3=IA en proceso, 4=IA ejecutado, 5=No Aplica\n")

t0 = time.time()
df_cruce_estado = query_to_df("""
SELECT 
    f.estado_glosa_aut AS estado,
    CASE 
        WHEN d.id IS NOT NULL THEN 2
        WHEN EXISTS (SELECT 1 FROM cm_detalles cd 
                     INNER JOIN cm_auditoria_motivos_glosas g ON g.cm_detalles_id = cd.id 
                     WHERE cd.cm_facturas_id = f.id) THEN 1
        ELSE 0 
    END AS target_real,
    COUNT(*) AS n
FROM cm_facturas f
LEFT JOIN cm_auditoria_devoluciones d ON d.cm_facturas_id = f.id
WHERE f.fecha_radicacion >= '2024-03-01' AND f.fecha_radicacion < '2026-01-01'
GROUP BY f.estado_glosa_aut, target_real
ORDER BY f.estado_glosa_aut, target_real
""")
print(f"  Query ejecutada en {time.time()-t0:.1f}s\n")

df_pivot_e = df_cruce_estado.pivot_table(
    index='estado', columns='target_real', values='n', 
    fill_value=0, aggfunc='sum'
)
df_pivot_e.columns = [LABELS.get(c, c) for c in df_pivot_e.columns]
df_pivot_e['Total'] = df_pivot_e.sum(axis=1)
for col in ['Auditada', 'Glosada', 'Devuelta']:
    if col in df_pivot_e.columns:
        df_pivot_e[f'%_{col}'] = (df_pivot_e[col] / df_pivot_e['Total'] * 100).round(1)

ESTADO_LABELS = {0: 'Pendiente', 1: 'Bot en proceso', 2: 'Bot ejecutado', 
                 3: 'IA en proceso', 4: 'IA ejecutado', 5: 'No Aplica'}
df_pivot_e.index = [f"{ESTADO_LABELS.get(idx, idx)} (={idx})" for idx in df_pivot_e.index]

print("── Tabla de contingencia: estado bot/IA vs resultado real ──\n")
print(df_pivot_e.to_string())

In [ ]:
# Celda 6 — Cruce combinado: ¿Qué dice el bot (código FA) + estado, cruzado con target?
# Para las facturas donde el bot SÍ ejecutó (estado_glosa_aut IN (2,4)),
# ¿qué código FA asignó y cuál fue el resultado real?

print("=" * 70)
print("DETALLE: Facturas donde bot/IA EJECUTÓ (estado 2 o 4)")
print("=" * 70)

t0 = time.time()
df_detalle = query_to_df("""
SELECT 
    f.ejecucion_glosa_aut_ia AS codigo_ia,
    f.estado_glosa_aut AS estado,
    CASE 
        WHEN d.id IS NOT NULL THEN 2
        WHEN EXISTS (SELECT 1 FROM cm_detalles cd 
                     INNER JOIN cm_auditoria_motivos_glosas g ON g.cm_detalles_id = cd.id 
                     WHERE cd.cm_facturas_id = f.id) THEN 1
        ELSE 0 
    END AS target_real,
    COUNT(*) AS n,
    ROUND(AVG(f.valor_factura), 0) AS valor_promedio
FROM cm_facturas f
LEFT JOIN cm_auditoria_devoluciones d ON d.cm_facturas_id = f.id
WHERE f.fecha_radicacion >= '2024-03-01' AND f.fecha_radicacion < '2026-01-01'
  AND f.estado_glosa_aut IN (2, 4)
GROUP BY f.ejecucion_glosa_aut_ia, f.estado_glosa_aut, target_real
ORDER BY f.ejecucion_glosa_aut_ia, f.estado_glosa_aut, target_real
""")
print(f"  Query ejecutada en {time.time()-t0:.1f}s\n")

if len(df_detalle) > 0:
    for estado_val in [2, 4]:
        estado_label = ESTADO_LABELS.get(estado_val, str(estado_val))
        sub = df_detalle[df_detalle['estado'] == estado_val]
        if len(sub) > 0:
            print(f"\n── Estado: {estado_label} (={estado_val}) ──")
            for _, row in sub.iterrows():
                fa = FA_CODES.get(int(row['codigo_ia']), f"Código {row['codigo_ia']}")
                target_label = LABELS.get(int(row['target_real']), str(row['target_real']))
                print(f"  {fa} → {target_label}: {int(row['n']):>8,} facturas (valor prom: ${int(row['valor_promedio']):>12,} COP)")
else:
    print("⚠️ No hay facturas con estado_glosa_aut IN (2, 4)")

# Resumen: cobertura del bot sobre el universo de facturas glosadas
print(f"\n{'='*70}")
print("COBERTURA DEL BOT sobre facturas REALMENTE glosadas")
print("=" * 70)

t0 = time.time()
df_cobertura = query_to_df("""
SELECT 
    CASE WHEN f.ejecucion_glosa_aut_ia > 0 THEN 'Bot detectó FA' ELSE 'Sin detección' END AS tipo,
    COUNT(*) AS n
FROM cm_facturas f
WHERE f.fecha_radicacion >= '2024-03-01' AND f.fecha_radicacion < '2026-01-01'
  AND EXISTS (SELECT 1 FROM cm_detalles cd 
              INNER JOIN cm_auditoria_motivos_glosas g ON g.cm_detalles_id = cd.id 
              WHERE cd.cm_facturas_id = f.id)
  AND NOT EXISTS (SELECT 1 FROM cm_auditoria_devoluciones d WHERE d.cm_facturas_id = f.id)
GROUP BY CASE WHEN f.ejecucion_glosa_aut_ia > 0 THEN 'Bot detectó FA' ELSE 'Sin detección' END
""")
print(f"  ({time.time()-t0:.1f}s)")

if len(df_cobertura) > 0:
    total_glosadas = int(df_cobertura['n'].sum())
    detectadas = int(df_cobertura[df_cobertura['tipo'] == 'Bot detectó FA']['n'].sum()) if 'Bot detectó FA' in df_cobertura['tipo'].values else 0
    recall_bot = detectadas / total_glosadas * 100 if total_glosadas > 0 else 0
    print(f"\n  Total facturas glosadas: {total_glosadas:,}")
    print(f"  Detectadas por bot (código FA > 0): {detectadas:,}")
    print(f"  Recall del bot para Glosadas: {recall_bot:.1f}%")
    print(f"  No detectadas: {total_glosadas - detectadas:,} ({100 - recall_bot:.1f}%)")

In [ ]:
# Celda 7 — Muestra de facturas con códigos FA para entender el patrón
# Ver ejemplos concretos de facturas donde el bot detectó FA

print("=" * 70)
print("MUESTRA DE FACTURAS con ejecucion_glosa_aut_ia > 0")
print("=" * 70)

t0 = time.time()
df_muestra = query_to_df("""
SELECT 
    f.id,
    f.ejecucion_glosa_aut_ia,
    f.estado_glosa_aut,
    f.estado_factura,
    f.valor_factura,
    f.fecha_radicacion,
    f.mae_tipo_contrato_valor AS tipo_contrato,
    CASE 
        WHEN d.id IS NOT NULL THEN 'Devuelta'
        WHEN EXISTS (SELECT 1 FROM cm_detalles cd 
                     INNER JOIN cm_auditoria_motivos_glosas g ON g.cm_detalles_id = cd.id 
                     WHERE cd.cm_facturas_id = f.id) THEN 'Glosada'
        ELSE 'Auditada' 
    END AS target_real
FROM cm_facturas f
LEFT JOIN cm_auditoria_devoluciones d ON d.cm_facturas_id = f.id
WHERE f.ejecucion_glosa_aut_ia > 0
  AND f.fecha_radicacion >= '2024-03-01'
ORDER BY f.ejecucion_glosa_aut_ia, RAND()
LIMIT 30
""")
print(f"  ({time.time()-t0:.1f}s)\n")

if len(df_muestra) > 0:
    # Mostrar agrupado por código FA
    for codigo in sorted(df_muestra['ejecucion_glosa_aut_ia'].unique()):
        fa_label = FA_CODES.get(int(codigo), f"Código {codigo}")
        sub = df_muestra[df_muestra['ejecucion_glosa_aut_ia'] == codigo]
        print(f"\n── {fa_label} (código={int(codigo)}) — {len(sub)} ejemplos ──")
        for _, row in sub.iterrows():
            print(f"  ID={int(row['id']):>10}  estado_glosa_aut={int(row['estado_glosa_aut'])}  "
                  f"estado_factura={int(row['estado_factura'])}  "
                  f"valor=${int(row['valor_factura']):>12,}  "
                  f"fecha={row['fecha_radicacion']}  "
                  f"contrato={row['tipo_contrato']}  "
                  f"→ {row['target_real']}")
else:
    print("⚠️ No hay facturas con ejecucion_glosa_aut_ia > 0")

print(f"\n{'='*70}")
print("CONCLUSIONES PRELIMINARES")
print("=" * 70)
print("""
Basado en los datos anteriores, evaluar:

1. Si ejecucion_glosa_aut_ia > 0 correlaciona fuertemente con target=Glosada,
   entonces el mapeo correcto es: FA > 0 → predicción "Glosada", FA = 0 → "Sin predicción"
   (NO "Auditada", porque el bot no procesó esas facturas)

2. Si el bot tiene baja cobertura (pocas facturas procesadas), la comparación
   solo es válida en el subconjunto donde el bot SÍ ejecutó.

3. El mapeo anterior (texto 'glos'/'devol') era incorrecto porque los valores
   son numéricos (0-5), no strings.
""")